# 原子分辨 STM 图像缺陷分割 — SAM 3（自动阈值）

使用 SAM 3 对原子分辨表面图像进行缺陷自动分割，包括：
- **CDW 超结构 / 表面重构**（十字星形亮斑）
- **点缺陷 / 空位**（暗斑点）

**自动确定阈值**，无需手动调参。

## 1. 环境准备


In [ ]:
import sys, os, numpy as np
from pathlib import Path
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Circle
import torch

# 中文字体（Windows 兼容）
for _f in matplotlib.font_manager.findSystemFonts():
    if any(k in _f for k in ("NotoSansCJK", "NotoSerifCJK", "msyh", "SimHei", "Microsoft YaHei")):
        matplotlib.rcParams["font.family"] = matplotlib.font_manager.FontProperties(fname=_f).get_name()
        break
matplotlib.rcParams["axes.unicode_minus"] = False

# ── 项目路径 ──────────────────────────────────────────────
def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "lumen").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the Lumen repository.")

PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
SAM3_ROOT = str(PROJECT_ROOT / "SEM zero-shot")
sys.path.insert(0, SAM3_ROOT)

print(f"Torch: {torch.__version__}, CUDA: {torch.cuda.is_available()}")
print(f"Project: {PROJECT_ROOT}")

## 2. 构建 SAM 3


In [ ]:
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

CHECKPOINT_PATH = str(PROJECT_ROOT / "checkpoints" / "sam3" / "sam3.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("Building SAM 3...")
model = build_sam3_image_model(
    checkpoint_path=CHECKPOINT_PATH, device=DEVICE,
    eval_mode=True, enable_segmentation=True, enable_inst_interactivity=False,
)
print(f"Parameters: {sum(p.numel() for p in model.parameters())/1e6:.0f}M")

# ── 修复 BFloat16/Float dtype 不匹配 ─────────────────────
if DEVICE == "cuda":
    import sam3.model.vitdet as _vitdet
    _original_addmm_act = _vitdet.addmm_act
    def _patched_addmm_act(activation, linear, mat1):
        orig_dtype = mat1.dtype
        return _original_addmm_act(activation, linear, mat1).to(orig_dtype)
    _vitdet.addmm_act = _patched_addmm_act
    print("Applied addmm_act dtype patch")

# 关键: threshold=0 获取全部 200 个预测
processor = Sam3Processor(model, resolution=1008, device=DEVICE, confidence_threshold=0.0)
print("Processor ready")

## 3. 加载图像目录

批量处理 `png-modulation` 下全部 20 张 FeTe STM 图像。

In [ ]:
# ============================================================
#  图像目录 & 裁切参数
# ============================================================
IMAGE_DIR = PROJECT_ROOT / "data" / "stm_dataset" / "FeTe-sxm" / "png-modulation"
OUT_DIR   = PROJECT_ROOT / "SEM zero-shot" / "output-sam3"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CROP_TOP    = 50   # 顶部裁切像素数（标尺区域），设为 0 则不裁切
CROP_BOTTOM = 0
CROP_LEFT   = 0
CROP_RIGHT  = 0

image_paths = sorted(IMAGE_DIR.glob("*.png"))
print(f"共找到 {len(image_paths)} 张图像:")
for p in image_paths:
    print(f"  {p.name}")


def load_and_crop(path):
    """加载单张图像，缩放 + 裁切，返回 PIL Image"""
    img = Image.open(path).convert("RGB")
    if max(img.size) > 1008:
        img.thumbnail((1008, 1008), Image.LANCZOS)
    if CROP_TOP > 0 or CROP_BOTTOM > 0 or CROP_LEFT > 0 or CROP_RIGHT > 0:
        w, h = img.size
        img = img.crop((CROP_LEFT, CROP_TOP, w - CROP_RIGHT, h - CROP_BOTTOM))
    return img

## 4. 多 prompt 推理 + 自动阈值

原子分辨图像包含**两种不同尺度的物理特征**，各自需要专门的文本 prompt：

| 特征 | Prompt | 描述 |
|------|--------|------|
| CDW 超结构 | `a bright cross dot` | 十字交叉状亮斑 |
| 点缺陷 | `a dark dot` | 亮背景上的暗圆孔 |

设置 `confidence_threshold=0` 拿到全部 200 个预测，再对每种 prompt 独立用肘部法确定最佳截断点。

In [ ]:
import cv2
import time

def auto_threshold_filter(scores, masks, boxes, skip_top=1, elbow_shift=0):
    s = np.sort(scores)[::-1]
    s_rest = s[skip_top:]
    drops = [s_rest[i] - s_rest[i+1] for i in range(min(30, len(s_rest)-1))]
    elbow = np.argmax(drops) + 1
    elbow_adj = int(np.clip(elbow + elbow_shift, 1, len(s_rest) - 1))
    thresh = s_rest[elbow_adj]
    keep = scores > thresh
    return masks[keep], boxes[keep], scores[keep], thresh


def multi_prompt_segment(processor, img_pil, prompt_configs, verbose=True):
    results = {}
    for cfg in prompt_configs:
        state = processor.set_image(img_pil)
        output = processor.set_text_prompt(prompt=cfg["prompt"], state=state)
        scores_np = output["scores"].cpu().numpy()
        masks_f, boxes_f, scores_f, thresh = auto_threshold_filter(
            scores_np, output["masks"], output["boxes"],
            skip_top=0, elbow_shift=cfg["elbow_shift"],
        )
        if verbose:
            print(f"    {cfg['name']}: {len(masks_f)} masks  (thresh={thresh:.4f})")
        results[cfg["name"]] = {
            "masks": masks_f, "boxes": boxes_f, "scores": scores_f,
            "threshold": thresh, "prompt": cfg["prompt"], "desc": cfg["desc"],
            "key": cfg["key"], "color": cfg["color"],
            "min_area_ratio": cfg.get("min_area_ratio", 0),
            "max_area_ratio": cfg.get("max_area_ratio", 0.20),
        }
        if DEVICE == "cuda":
            torch.cuda.empty_cache()
    return results


def extract_features(feature_masks, image_np):
    gray = image_np.mean(axis=2).astype(np.float32) if image_np.ndim == 3 else image_np.astype(np.float32)
    all_features = {}
    for fname, masks_list in feature_masks.items():
        spots = []
        for idx, mask in enumerate(masks_list):
            area = int(cv2.countNonZero(mask))
            if area == 0:
                continue
            M = cv2.moments(mask, binaryImage=True)
            cx = M["m10"] / M["m00"]
            cy = M["m01"] / M["m00"]
            masked_gray = gray * mask.astype(np.float32)
            intensity_sum = float(masked_gray.sum())
            peak = float(cv2.minMaxLoc(gray, mask=mask)[1])
            spots.append({
                "index": idx, "cx": cx, "cy": cy,
                "radius": float(np.sqrt(area / np.pi)),
                "area": area,
                "intensity_mean": intensity_sum / area,
                "peak": peak,
            })
        all_features[fname] = spots
    return all_features


def masks_to_numpy(all_results, h, w):
    total_pixels = h * w
    feature_masks = {}
    for name, res in all_results.items():
        masks_t = res["masks"]
        if hasattr(masks_t, "cpu"):
            masks_np = masks_t.cpu().numpy().squeeze()
        else:
            masks_np = np.array(masks_t).squeeze()
        if masks_np.ndim == 4:
            masks_np = masks_np[:, 0]
        if masks_np.ndim == 2:
            masks_np = masks_np[np.newaxis, :, :]
        min_px = int(total_pixels * res.get("min_area_ratio", 0))
        max_px = int(total_pixels * res.get("max_area_ratio", 0.20))
        resized = []
        n_too_big = 0
        n_too_small = 0
        for m in masks_np:
            if m.shape != (h, w):
                m = cv2.resize(m.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
            m = np.squeeze(m).astype(np.uint8)
            area = cv2.countNonZero(m)
            if area > max_px:
                n_too_big += 1; continue
            if area < min_px:
                n_too_small += 1; continue
            resized.append(m)
        if n_too_big:
            print(f"    ⚠ {name}: 过滤 {n_too_big} 个过大 mask")
        if n_too_small:
            print(f"    ⚠ {name}: 过滤 {n_too_small} 个过小 mask")
        feature_masks[name] = resized
    return feature_masks


def merge_overlapping_masks(feature_masks, min_area=10, verbose=False):
    """Union 同类所有 mask, connectedComponents 拆成独立特征。
    
    200 个重叠 mask 变成 N 个真实连通区域 = 真实物理特征数。
    min_area: 连通区域最小像素数，低于此值丢弃（噪声）。
    """
    merged = {}
    for name, masks_list in feature_masks.items():
        if not masks_list:
            merged[name] = []
            continue
        h, w = masks_list[0].shape
        composite = np.zeros((h, w), dtype=np.uint8)
        for m in masks_list:
            composite = np.maximum(composite, m)
        n_labels, labels = cv2.connectedComponents(composite, connectivity=8)
        new_masks = []
        for lid in range(1, n_labels):
            component = (labels == lid).astype(np.uint8)
            if cv2.countNonZero(component) < min_area:
                continue
            new_masks.append(component)
        if verbose:
            print(f"    merge {name}: {len(masks_list)} masks -> {len(new_masks)} features")
        merged[name] = new_masks
    return merged


_LEGEND_LABELS = {
    "cdw_superstructure": "CDW Superstructure",
    "point_defect":       "Point Defect",
}

def _draw_legend(img, entries, margin=10, swatch=14, line_h=22,
                 font_scale=0.50, thickness=1):
    if not entries:
        return
    font = cv2.FONT_HERSHEY_SIMPLEX
    h, w = img.shape[:2]
    labels = []
    for e in entries:
        lb = _LEGEND_LABELS.get(e["key"], e["key"])
        if e.get("count") is not None:
            lb += f" ({e['count']})"
        labels.append(lb)
    max_tw = max(cv2.getTextSize(lb, font, font_scale, thickness)[0][0] for lb in labels)
    box_w = margin + swatch + 8 + max_tw + margin
    box_h = margin + line_h * len(entries) + margin // 2
    x0 = w - box_w - margin
    y0 = h - box_h - margin
    overlay = img.copy()
    cv2.rectangle(overlay, (x0, y0), (w - margin, h - margin), (0, 0, 0), -1)
    cv2.addWeighted(overlay, 0.55, img, 0.45, 0, dst=img)
    for i, (e, label) in enumerate(zip(entries, labels)):
        cy = y0 + margin + i * line_h
        sx = x0 + margin
        cv2.rectangle(img, (sx, cy), (sx + swatch, cy + swatch), e["color"], -1)
        cv2.rectangle(img, (sx, cy), (sx + swatch, cy + swatch), (255, 255, 255), 1)
        cv2.putText(img, label, (sx + swatch + 8, cy + swatch - 2),
                    font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)


def save_overlay(img_rgb, all_results, feature_masks, save_path, alpha=0.45):
    vis = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    h, w = vis.shape[:2]
    for name, res in all_results.items():
        color = res["color"]
        masks_list = feature_masks.get(name, [])
        if not masks_list:
            continue
        composite = np.zeros((h, w), dtype=np.uint8)
        for mask in masks_list:
            composite = np.maximum(composite, mask)
        colored = vis.copy()
        colored[composite > 0] = color
        cv2.addWeighted(colored, alpha, vis, 1 - alpha, 0, dst=vis)
        contours, _ = cv2.findContours(composite, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, contours, -1, color, 1)
    legend_entries = [
        {"key": res["key"], "color": res["color"], "count": len(feature_masks.get(name, []))}
        for name, res in all_results.items()
    ]
    _draw_legend(vis, legend_entries)
    cv2.imwrite(str(save_path), vis, [cv2.IMWRITE_PNG_COMPRESSION, 1])


# ═══════════════════════════════════════════════════════════
prompt_configs = [
    {
        "name": "CDW超结构",
        "key": "cdw_superstructure",
        "prompt": "a bright cross dot",
        "elbow_shift": 300,
        "color": (255, 0, 255),
        "desc": "CDW 调制原子",
        "max_area_ratio": 0.005,
    },
    {
        "name": "点缺陷",
        "key": "point_defect",
        "prompt": "a dark dot",
        "elbow_shift": 300,
        "color": (0, 0, 255),
        "desc": "点缺陷",
        "max_area_ratio": 0.01,
    },
]

print(f"已定义 {len(prompt_configs)} 种特征 prompt")
print(f"重叠 mask 自动合并 (connectedComponents)")

## 5. 批量推理 + 保存

遍历全部 20 张图像，逐张执行两 prompt 分割 → 提取定量参数。

**保存控制**：设置 `SAVE_OVERLAY / SAVE_JSON_CSV = True/False` 控制是否保存文件。

In [ ]:
import json, csv

SAVE_OVERLAY  = True
SAVE_JSON_CSV = True

all_summaries = []
all_batch_results = {}
total_t0 = time.time()

for img_idx, img_path in enumerate(image_paths):
    stem = img_path.stem
    print(f"\n[{img_idx+1}/{len(image_paths)}] {stem}", end="")

    img_pil = load_and_crop(img_path)
    img_np = np.array(img_pil)
    h, w = img_np.shape[:2]

    t0 = time.time()
    all_results = multi_prompt_segment(processor, img_pil, prompt_configs, verbose=False)
    t_infer = time.time() - t0

    t0 = time.time()
    feature_masks = masks_to_numpy(all_results, h, w)
    # ★ 合并重叠 mask: 200 个 SAM mask -> N 个真实连通特征
    feature_masks = merge_overlapping_masks(feature_masks, min_area=10, verbose=True)
    all_features = extract_features(feature_masks, img_np)
    t_post = time.time() - t0

    all_batch_results[stem] = {
        "img_np": img_np, "all_results": all_results,
        "feature_masks": feature_masks, "all_features": all_features,
    }

    t0 = time.time()
    if SAVE_OVERLAY:
        save_overlay(img_np, all_results, feature_masks,
                     OUT_DIR / f"{stem}_overlay.png")
    if SAVE_JSON_CSV:
        for fname, spots in all_features.items():
            safe = fname.replace("/", "_")
            with open(OUT_DIR / f"{stem}_{safe}.json", "w", encoding="utf-8") as f:
                json.dump(spots, f, indent=2, ensure_ascii=False)
            if spots:
                with open(OUT_DIR / f"{stem}_{safe}.csv", "w", newline="") as f:
                    w_ = csv.DictWriter(f, fieldnames=list(spots[0].keys()))
                    w_.writeheader()
                    w_.writerows(spots)
    t_save = time.time() - t0

    n_masks = sum(len(v) for v in feature_masks.values())
    print(f"  -> {n_masks} features | infer {t_infer:.1f}s  post {t_post:.2f}s  save {t_save:.2f}s")

    summary = {
        "image": stem, "size": [w, h],
        "n_features": {k: len(v) for k, v in all_features.items()},
        "thresholds": {k: float(all_results[k]["threshold"]) for k in all_results},
    }
    all_summaries.append(summary)

if SAVE_JSON_CSV:
    with open(OUT_DIR / "batch_summary.json", "w", encoding="utf-8") as f:
        json.dump(all_summaries, f, indent=2, ensure_ascii=False)

total_time = time.time() - total_t0
print(f"\n{'#'*50}")
print(f"Done: {len(all_summaries)} images, {total_time:.1f}s total")
print(f"Output: {OUT_DIR}")
print(f"{'#'*50}")

## 6. 批量统计汇总

各特征在 20 张图上的检测数量分布。

In [ ]:
# ── 汇总表格 ──────────────────────────────────────────────
print(f"{'Image':<14}", end="")
for cfg in prompt_configs:
    print(f"  {cfg['name']:<8}", end="")
print(f"  {'Total':<6}")
print("-" * 52)

for s in all_summaries:
    print(f"{s['image']:<14}", end="")
    row_total = 0
    for cfg in prompt_configs:
        n = s["n_features"].get(cfg["name"], 0)
        row_total += n
        print(f"  {n:<8}", end="")
    print(f"  {row_total:<6}")

# ── 均值 ──────────────────────────────────────────────────
print("-" * 52)
print(f"{'Mean':<14}", end="")
for cfg in prompt_configs:
    vals = [s["n_features"].get(cfg["name"], 0) for s in all_summaries]
    print(f"  {np.mean(vals):<8.1f}", end="")
total_vals = [sum(s["n_features"].values()) for s in all_summaries]
print(f"  {np.mean(total_vals):<6.1f}")

## 7. 全部结果展示

展示 20 张图像的叠加分割结果（CDW + 点缺陷）。设置 `SAVE_FIGURE = True` 可保存为 PNG。

In [ ]:
# ══════════════ 保存控制 ═════════════════════════════════
SAVE_FIGURE = False   # 改为 True 即可保存汇总大图
FIGURE_PATH = OUT_DIR / "all_overlays_summary.png"
# ═════════════════════════════════════════════════════════

all_overlay_paths = sorted(OUT_DIR.glob("*_overlay.png"))
n = len(all_overlay_paths)
cols = 4
rows = (n + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(cols * 5, rows * 5))
axes = axes.flat

for ax, p in zip(axes, all_overlay_paths):
    ax.imshow(Image.open(p))
    ax.set_title(p.stem.replace("_overlay", ""), fontsize=11)
    ax.axis("off")

# 隐藏多余的空白子图
for ax in list(axes)[n:]:
    ax.axis("off")

plt.suptitle(f"SAM3 多特征分割 — 全部 {n} 张结果", fontsize=15, y=1.0)
plt.tight_layout()

if SAVE_FIGURE:
    fig.savefig(str(FIGURE_PATH), dpi=150, bbox_inches="tight")
    print(f"已保存汇总图: {FIGURE_PATH}")
else:
    print("（如需保存汇总图，设置上方 SAVE_FIGURE = True 后重新运行此 cell）")

plt.show()

## 8. 总结

批量处理 20 张 FeTe STM 图像的全自动缺陷分割：

| 特征 | Prompt | 自动阈值 | 物理意义 |
|------|--------|---------|---------|
| CDW 超结构 | `a bright cross dot` | 肘部法 (shift=200) | 电荷密度波 / 表面重构 |
| 点缺陷 | `a dark dot` | 肘部法 (shift=200) | 原子空位 / 杂质 |

**每张图输出：**
- `{stem}_overlay.png` — 两特征轮廓叠加图
- `{stem}_{特征名}.json / .csv` — 定量参数（中心坐标、面积、强度等）
- `batch_summary.json` — 全局汇总统计